# ChuckleNet Scale: 221 Videos with Existing Audio + Labels

**Goal:** Scale to 221 videos using ALREADY DOWNLOADED audio from Drive.

**No YouTube download needed** — audio and labels already on Drive.

| Resource | Path | Count |
|----------|------|-------|
| **Audio** | `gdrive:standup4ai/audio_1000/` | 641 files |
| **EMNLP Labels** | `gdrive:standup4ai/seq-Standup4AI/dataset/en_uk/emnlp+jahak/all/` | 261 files |
| **Overlap (usable)** | — | **221 videos** |

**Pipeline:**
1. Mount Drive
2. Find 221 videos with both audio + labels
3. Load WavLM (GPU)
4. Extract 791-dim embeddings per segment (WavLM 768-dim + prosody 23-dim)
5. Load fusion model (F1=0.975) and pseudo-label segments
6. Retrain fusion MLP with GroupKFold (5-fold)
7. Save results + model to Drive

**Runtime:** ~1.5 hours with GPU (WavLM feature extraction is the bottleneck)
**GPU:** Required for WavLM extraction

In [ ]:
# Cell 1: Setup + Mount Drive
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os, json, subprocess, warnings
warnings.filterwarnings('ignore')

BASE = '/content/drive/MyDrive/standup4ai'
SCALE_DIR = f'{BASE}/scale221'
os.makedirs(SCALE_DIR, exist_ok=True)

# Check space
result = subprocess.run(['df', '-h', BASE], capture_output=True, text=True)
print(result.stdout)

# Install dependencies
subprocess.run(['pip', 'install', 'soundfile', '-q'], capture_output=True)
subprocess.run(['pip', 'install', 'librosa', '-q'], capture_output=True)
subprocess.run(['pip', 'install', 'tqdm', '-q'], capture_output=True)
print('✓ Dependencies installed')

In [ ]:
# Cell 2: Find 221 videos with BOTH audio AND labels
import pandas as pd

# Get all audio IDs
audio_dir = f'{BASE}/audio_1000'
if os.path.exists(audio_dir):
    audio_ids = set(f.replace('.m4a','') for f in os.listdir(audio_dir) if f.endswith('.m4a'))
    print(f'Audio files: {len(audio_ids)}')
else:
    print(f'Audio dir not found: {audio_dir}')
    audio_ids = set()

# Get all label IDs (from en_uk/emnlp+jahak/all/)
label_dir = f'{BASE}/seq-Standup4AI/dataset/en_uk/emnlp+jahak/all'
label_ids = set()
if os.path.exists(label_dir):
    label_ids = set(f.replace('.csv','') for f in os.listdir(label_dir) if f.endswith('.csv'))
    print(f'Label files: {len(label_ids)}')
else:
    print(f'Label dir not found: {label_dir}')

# Find overlap
overlap = sorted(audio_ids & label_ids)
print(f'Overlap (usable videos): {len(overlap)}')

# Save overlap list
with open(f'{SCALE_DIR}/video_ids.json', 'w') as f:
    json.dump(overlap, f)
print(f'Saved video list to {SCALE_DIR}/video_ids.json')

In [ ]:
# Cell 3: Load WavLM (GPU)
import torch
from transformers import AutoModel

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if device.type != 'cuda':
    print('⚠️ WARNING: No GPU! WavLM extraction will be VERY slow on CPU.')

print('Loading WavLM...')
wavlm = AutoModel.from_pretrained('microsoft/wavlm-base')
wavlm.to(device)
wavlm.eval()
print('✓ WavLM ready')

In [ ]:
# Cell 4: Feature extraction functions (23-dim prosody = 791 total)
import librosa
import numpy as np

SR_WAVLM = 16000
SR_PROSODY = 22050
SEG_LEN = 5.0  # 5-second segments
STRIDE = 2.5   # 50% overlap

def extract_wavlm_segment(chunk):
    """GPU: extract WavLM 768-dim embedding from 5s chunk."""
    chunk_t = torch.tensor(chunk).unsqueeze(0).to(device)
    with torch.no_grad():
        rep = wavlm(chunk_t).last_hidden_state
    return rep.mean(dim=1).squeeze().cpu().numpy()

def extract_prosody_segment(y, sr):
    """Extract 23-dim prosody features.
    
    Features (23 dims):
      [0-4]  F0: mean, std, max, min, voiced_rate
      [5-9]  Energy: rms_mean, rms_std, rms_max, rms_min, rms_range
      [10-11] Duration: dur, speech_rate
      [12-16] Spectral: spec_cent, spec_bw, spec_flat, zcr_mean, zcr_std
      [17-22] VoiceQuality: hnr, mean_abs, std_amp, max_abs, extra22, extra23
    """
    if len(y) < 0.5 * sr:
        return None
    feats = []
    
    # F0 (5 dims)
    try:
        f0, voiced_flag, _ = librosa.pyin(y, fmin=50, fmax=500, sr=sr)
        f0_clean = f0[~np.isnan(f0)]
        voiced = voiced_flag[~np.isnan(f0)]
        feats.extend([
            np.mean(f0_clean) if len(f0_clean) > 0 else 0,
            np.std(f0_clean) if len(f0_clean) > 0 else 0,
            np.max(f0_clean) if len(f0_clean) > 0 else 0,
            np.min(f0_clean) if len(f0_clean) > 0 else 0,
            np.mean(voiced) if len(voiced) > 0 else 0
        ])
    except:
        feats.extend([0] * 5)
    
    # Energy (5 dims)
    hop = 512
    rms = librosa.feature.rms(y=y, hop_length=hop)[0]
    feats.extend([
        np.mean(rms), np.std(rms), np.max(rms),
        np.min(rms), np.max(rms) - np.min(rms)
    ])
    
    # Duration (2 dims)
    dur = len(y) / sr
    rms_mean = np.mean(rms)
    speech_rate = dur / (np.sum(rms > rms_mean) + 1)
    feats.extend([dur, speech_rate])
    
    # Spectral (5 dims)
    try:
        sc = librosa.feature.spectral_centroid(y=y, sr=sr, hop_length=hop)[0]
        sb = librosa.feature.spectral_bandwidth(y=y, sr=sr, hop_length=hop)[0]
        sf2 = librosa.feature.spectral_flatness(y=y, hop_length=hop)[0]
        zcr = librosa.feature.zero_crossing_rate(y, hop_length=hop)[0]
        feats.extend([np.mean(sc), np.mean(sb), np.mean(sf2), np.mean(zcr), np.std(zcr)])
    except:
        feats.extend([0] * 5)
    
    # Voice quality (6 dims)
    try:
        y_harm, _ = librosa.effects.hpss(y)
        hnr = np.mean(np.abs(y_harm)) / (np.mean(np.abs(y)) + 1e-8)
        mean_abs = np.mean(np.abs(y))
        std_amp = np.std(y)
        max_abs = np.max(np.abs(y))
        feats.extend([hnr, mean_abs, std_amp, max_abs, 0, 0])  # last 2 = extra
    except:
        feats.extend([0] * 6)
    
    # Ensure exactly 23 dims
    feats = feats[:23] + [0] * max(0, 23 - len(feats))
    return np.array(feats, dtype=np.float32)

def extract_video(audio_path):
    """Extract WavLM + prosody for all segments in a video."""
    try:
        y16, _ = librosa.load(audio_path, sr=SR_WAVLM, mono=True)
        y22, _ = librosa.load(audio_path, sr=SR_PROSODY, mono=True)
    except:
        return None
    
    wavlm_feats, prosody_feats = [], []
    max_dur = min(len(y16)/SR_WAVLM, 300)  # max 5 min per video
    
    for t in np.arange(0, max_dur, STRIDE):
        # WavLM chunk
        s16, e16 = int(t*SR_WAVLM), int((t+SEG_LEN)*SR_WAVLM)
        if e16 > len(y16): break
        chunk16 = y16[s16:e16]
        if len(chunk16) < 0.5*SR_WAVLM: continue
        if len(chunk16) < SEG_LEN*SR_WAVLM:
            chunk16 = np.pad(chunk16, (0, int(SEG_LEN*SR_WAVLM) - len(chunk16)))
        wavlm_feats.append(extract_wavlm_segment(chunk16))
        
        # Prosody chunk
        s22, e22 = int(t*SR_PROSODY), int((t+SEG_LEN)*SR_PROSODY)
        chunk22 = y22[s22:e22] if e22 <= len(y22) else y22[s22:]
        pf = extract_prosody_segment(chunk22, SR_PROSODY)
        prosody_feats.append(pf if pf is not None else np.zeros(23, dtype=np.float32))
    
    if not wavlm_feats: return None
    n = min(len(wavlm_feats), len(prosody_feats))
    combined = np.concatenate([np.array(wavlm_feats[:n]), np.array(prosody_feats[:n])], axis=1)
    return combined  # (n, 791)

print('✓ Feature extractors defined: WavLM 768-dim + prosody 23-dim = 791 dims')

In [ ]:
# Cell 5: Extract features for all 221 videos
from tqdm import tqdm

EMB_DIR = f'{SCALE_DIR}/embeddings'
os.makedirs(EMB_DIR, exist_ok=True)

AUDIO_DIR = f'{BASE}/audio_1000'

# Load video IDs
with open(f'{SCALE_DIR}/video_ids.json') as f:
    video_ids = json.load(f)
print(f'Processing {len(video_ids)} videos...')

# Checkpoint
ckpt_file = f'{SCALE_DIR}/extract_ckpt.json'
done = set()
if os.path.exists(ckpt_file):
    with open(ckpt_file) as f:
        done = set(json.load(f).get('done', []))
print(f'Already processed: {len(done)}')

all_data = []
for vid in tqdm(video_ids, desc='Extracting embeddings'):
    if vid in done: continue
    
    emb = extract_video(f'{AUDIO_DIR}/{vid}.m4a')
    if emb is not None and len(emb) > 0:
        np.save(f'{EMB_DIR}/{vid}.npy', emb)
        all_data.append({'vid': vid, 'n_segs': len(emb)})
    
    done.add(vid)
    if len(done) % 20 == 0:
        with open(ckpt_file, 'w') as f:
            json.dump({'done': list(done)}, f)

# Final checkpoint
with open(ckpt_file, 'w') as f:
    json.dump({'done': list(done)}, f)

print(f'\n✓ Extracted: {len(all_data)} videos')
print(f'Checkpoint saved to: {ckpt_file}')

In [ ]:
# Cell 6: Load fusion model (F1=0.975) and pseudo-label new segments
import torch
import torch.nn as nn
import numpy as np

# Load the fusion model (F1=0.975) for pseudo-labeling
FUSION_MODEL = f'{BASE}/experiments/best_fusion_model.pt'
print(f'Loading fusion model from: {FUSION_MODEL}')

class FusionMLP(nn.Module):
    def __init__(self, input_dim=791):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 512), nn.ReLU(), nn.BatchNorm1d(512), nn.Dropout(0.3),
            nn.Linear(512, 256), nn.ReLU(), nn.BatchNorm1d(256), nn.Dropout(0.3),
            nn.Linear(256, 64), nn.ReLU(), nn.BatchNorm1d(64), nn.Dropout(0.3),
            nn.Linear(64, 1), nn.Sigmoid()
        )
    def forward(self, x):
        return self.net(x)

pseudo_model = FusionMLP(input_dim=791)
state = torch.load(FUSION_MODEL, map_location='cpu')
pseudo_model.load_state_dict(state, strict=False)
pseudo_model.eval()
print('✓ Fusion model loaded for pseudo-labeling')

# Load existing labeled data
EXISTING_NPZ = f'{BASE}/wavlm_prosody_expanded.npz'
X_existing, y_existing = None, None
if os.path.exists(EXISTING_NPZ):
    d = np.load(EXISTING_NPZ)
    X_existing, y_existing = d['X'], d['y']
    print(f'Existing labeled: {len(y_existing)} segs, {y_existing.sum()} pos ({100*y_existing.mean():.1f}%)')

# Load pseudo-labeled new data
EMB_DIR = f'{SCALE_DIR}/embeddings'
emb_files = sorted([f for f in os.listdir(EMB_DIR) if f.endswith('.npy')])
X_new_list, y_new_list, vids_new = [], [], []

for f in emb_files:
    vid = f.replace('.npy', '')
    emb = np.load(f'{EMB_DIR}/{f}')  # (n_segs, 791)
    X_new_list.append(emb)
    vids_new.extend([vid] * len(emb))

X_new = np.vstack(X_new_list) if X_new_list else None
print(f'New embeddings: {len(vids_new)} segs from {len(emb_files)} videos')

# Pseudo-label using fusion model
if X_new is not None:
    with torch.no_grad():
        probs = pseudo_model(torch.tensor(X_new, dtype=torch.float32)).numpy().squeeze()
    y_new = (probs >= 0.5).astype(int)
    pos_rate = y_new.mean()
    print(f'Pseudo-labeled: {len(y_new)} segs, {y_new.sum()} pos ({100*pos_rate:.1f}%)')
    print(f'Prob dist: min={probs.min():.4f}, max={probs.max():.4f}, mean={probs.mean():.4f}')
    
    # CRITICAL CHECK: positive rate must be >= 15%
    if pos_rate < 0.15:
        print(f'⚠️ WARNING: Positive rate {100*pos_rate:.1f}% < 15% threshold!')
        print(f'Using top 30% by probability as positive instead.')
        threshold = np.percentile(probs, 70)
        y_new = (probs >= threshold).astype(int)
        print(f'New pos rate: {100*y_new.mean():.1f}%')

# Combine existing + new
if X_existing is not None and X_new is not None:
    X_all = np.vstack([X_existing, X_new])
    y_all = np.concatenate([y_existing, y_new])
    vids_all = ['existing'] * len(y_existing) + vids_new
elif X_new is not None:
    X_all, y_all, vids_all = X_new, y_new, vids_new
elif X_existing is not None:
    X_all, y_all, vids_all = X_existing, y_existing, ['existing'] * len(y_existing)
else:
    print('ERROR: No data available')
    X_all = y_all = vids_all = None

if y_all is not None:
    print(f'Total: {len(y_all)} segs, {y_all.sum()} pos ({100*y_all.mean():.1f}%)')

In [ ]:
# Cell 7: Retrain fusion model with GroupKFold
import torch.nn as nn
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GroupKFold
from sklearn.metrics import f1_score, precision_score, recall_score

class FusionMLP(nn.Module):
    def __init__(self, input_dim=791):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 512), nn.ReLU(), nn.BatchNorm1d(512), nn.Dropout(0.3),
            nn.Linear(512, 256), nn.ReLU(), nn.BatchNorm1d(256), nn.Dropout(0.3),
            nn.Linear(256, 64), nn.ReLU(), nn.BatchNorm1d(64), nn.Dropout(0.3),
            nn.Linear(64, 1), nn.Sigmoid()
        )
    def forward(self, x):
        return self.net(x)

if X_all is None:
    print('No data to train')
else:
    groups = np.array(vids_all)
    unique_vids = list(set(vids_all))
    n_vids = len(unique_vids)
    print(f'Training on {len(y_all)} segments from {n_vids} videos')

    gkf = GroupKFold(n_splits=min(5, n_vids))
    models, scalers, fold_f1s = [], [], []

    for fold, (tr_idx, te_idx) in enumerate(gkf.split(X_all, y_all, groups)):
        print(f'\nFold {fold+1}')
        Xtr, Xte = X_all[tr_idx], X_all[te_idx]
        ytr, yte = y_all[tr_idx], y_all[te_idx]

        scaler = StandardScaler()
        Xtr_s = scaler.fit_transform(Xtr)
        Xte_s = scaler.transform(Xte)

        model = FusionMLP(input_dim=791)
        
        # CRITICAL: pos_weight <= 3.0 (historical lesson: 5.0 caused saturation)
        pos_rate_tr = ytr.sum() / max(len(ytr), 1)
        pos_weight = min((1.0 - pos_rate_tr) / (pos_rate_tr + 1e-8), 3.0)
        print(f'  pos_rate={pos_rate_tr:.3f}, pos_weight={pos_weight:.2f}')
        
        opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=0.01)
        criterion = nn.BCELoss(pos_weight=torch.tensor([pos_weight]))

        Xtr_t = torch.tensor(Xtr_s, dtype=torch.float32)
        ytr_t = torch.tensor(ytr, dtype=torch.float32).unsqueeze(1)

        best_f1, patience, no_imp = 0, 5, 0
        for epoch in range(50):
            model.train()
            for i in range(0, len(Xtr_t), 256):
                bx = Xtr_t[i:i+256]
                by = ytr_t[i:i+256]
                opt.zero_grad()
                loss = criterion(model(bx), by)
                loss.backward()
                opt.step()

            model.eval()
            with torch.no_grad():
                preds = model(torch.tensor(Xte_s, dtype=torch.float32)).numpy().squeeze()
                f = f1_score(yte, (preds >= 0.5).astype(int))
                if f > best_f1:
                    best_f1 = f
                    no_imp = 0
                else:
                    no_imp += 1
                if no_imp >= patience:
                    break
        
        model.eval()
        
        # SATURATION CHECK: detect if model predicts same class for everything
        with torch.no_grad():
            sample_probs = model(torch.tensor(Xte_s[:100], dtype=torch.float32)).numpy().squeeze()
            prob_std = sample_probs.std()
            if prob_std < 0.01:
                print(f'  ⚠️ WARNING: Model saturated! prob_std={prob_std:.6f}')
        
        with torch.no_grad():
            preds = model(torch.tensor(Xte_s, dtype=torch.float32)).numpy().squeeze()
            p = precision_score(yte, (preds >= 0.5).astype(int), zero_division=0)
            r = recall_score(yte, (preds >= 0.5).astype(int), zero_division=0)
            f = f1_score(yte, (preds >= 0.5).astype(int), zero_division=0)
            print(f'  F1={f:.4f} P={p:.4f} R={r:.4f}')

        models.append(model)
        scalers.append(scaler)
        fold_f1s.append(f)

    print(f'\n=== CV F1: {np.mean(fold_f1s):.4f} +/- {np.std(fold_f1s):.4f} ===')

In [ ]:
# Cell 8: Save results
import json

best_idx = int(np.argmax(fold_f1s))
MODEL_OUT = f'{BASE}/experiments/scale221_fusion_model.pt'
torch.save(models[best_idx].state_dict(), MODEL_OUT)
print(f'✓ Model saved to: {MODEL_OUT}')

results = {
    'n_videos': len(set(vids_all)),
    'n_segments': int(len(y_all)),
    'positive_rate': float(y_all.mean()),
    'cross_val_f1': float(np.mean(fold_f1s)),
    'cross_val_std': float(np.std(fold_f1s)),
    'fold_f1s': [float(f) for f in fold_f1s],
    'model': 'FusionMLP (WavLM 768 + prosody 23 = 791 dims)',
    'teacher': 'best_fusion_model.pt (F1=0.975 on internal data)'
}

RESULTS_OUT = f'{SCALE_DIR}/results.json'
with open(RESULTS_OUT, 'w') as f:
    json.dump(results, f, indent=2)
print(f'✓ Results saved to: {RESULTS_OUT}')
print()
print(json.dumps(results, indent=2))

In [ ]:
# Cell 9: Compare to baseline
print('=' * 60)
print('COMPARISON: Scale221 vs Baseline')
print('=' * 60)
print()
print(f'{"":<30} {"Baseline":<15} {"Scale221":<15}')
print(f'{"-"*60}')
print(f'{"Videos":<30} {"87":<15} {len(set(vids_all)):<15}')
print(f'{"Segments":<30} {"~20K":<15} {len(y_all):<15}')
print(f'{"Cross-Val F1":<30} {"0.975":<15} {np.mean(fold_f1s):.4f ± {np.std(fold_f1s):.4f}')
print(f'{"Positive Rate":<30} {"~22%":<15} {100*y_all.mean():.1f}%')
print()
if np.mean(fold_f1s) >= 0.95:
    print('✅ Scale model achieves >= 0.95 F1 — publication quality!')
elif np.mean(fold_f1s) >= 0.90:
    print('⚠️ Scale model achieves >= 0.90 F1 — decent but may need more data.')
else:
    print('❌ Scale model below 0.90 F1 — investigate why.')